In [4]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

import mlflow

PROJECT = r"D:\projects\real 3 JD prjct\churn-mlops-pipeline"
# Use a sqlite database — required for the model registry on Windows
mlflow.set_tracking_uri(f"sqlite:///{os.path.join(PROJECT, 'mlflow.db')}")
print("TRACKING URI =", mlflow.get_tracking_uri())

from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, recall_score,
                             precision_score, f1_score, confusion_matrix)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

mlflow.set_experiment("churn-prediction")
RANDOM_STATE = 42

TRACKING URI = sqlite:///D:\projects\real 3 JD prjct\churn-mlops-pipeline\mlflow.db


2026/07/26 14:33:24 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/26 14:33:24 INFO mlflow.store.db.utils: Updating database tables
2026/07/26 14:33:25 INFO mlflow.tracking.fluent: Experiment with name 'churn-prediction' does not exist. Creating a new experiment.


In [5]:
df = pd.read_csv(os.path.join(PROJECT, "data", "telco_churn.csv"))
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
print("rows:", len(df), "| churn rate:", f"{df['Churn'].mean():.2%}")

X = df.drop(columns=["Churn"])
y = df["Churn"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
X[cat_cols] = X[cat_cols].fillna("Unknown")
X = pd.get_dummies(X, columns=cat_cols, drop_first=False)
X[X.select_dtypes(include="bool").columns] = X[X.select_dtypes(include="bool").columns].astype(int)
X.columns = [c.replace(" ", "_").replace("-", "_") for c in X.columns]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
sw = (y_tr == 0).sum() / (y_tr == 1).sum()
print("features:", X_tr.shape[1], "| train:", X_tr.shape[0], "| test:", X_te.shape[0])

rows: 7043 | churn rate: 26.54%
features: 45 | train: 5634 | test: 1409


In [6]:
def train_and_log(model, params, name):
    with mlflow.start_run(run_name=name):
        mlflow.log_params(params)
        model.fit(X_tr, y_tr)
        y_pred  = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1]
        metrics = {
            "accuracy": accuracy_score(y_te, y_pred),
            "roc_auc": roc_auc_score(y_te, y_proba),
            "recall_churn": recall_score(y_te, y_pred, pos_label=1),
            "precision_churn": precision_score(y_te, y_pred, pos_label=1),
            "f1_churn": f1_score(y_te, y_pred, pos_label=1),
        }
        mlflow.log_metrics(metrics)
        fig, ax = plt.subplots(figsize=(4, 3))
        sns.heatmap(confusion_matrix(y_te, y_pred), annot=True, fmt="d", cmap="Blues",
                    xticklabels=["Stay", "Churn"], yticklabels=["Stay", "Churn"], ax=ax)
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
        mlflow.log_figure(fig, "confusion_matrix.png"); plt.close(fig)
        sig = infer_signature(X_tr, model.predict(X_tr))
        mlflow.sklearn.log_model(model, name="model", signature=sig,
                                 input_example=X_tr.iloc[:1],
                                 serialization_format="pickle")
        run_id = mlflow.active_run().info.run_id
        try:
            mlflow.register_model(model_uri=f"runs:/{run_id}/model", name="churn_model")
        except Exception as e:
            print("  (register note:", e, ")")
        print(f"[{name}] AUC={metrics['roc_auc']:.4f}  recall_churn={metrics['recall_churn']:.2f}")
        return metrics["roc_auc"], run_id

In [7]:
results = {}
xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, scale_pos_weight=sw,
                    random_state=RANDOM_STATE, eval_metric="logloss", use_label_encoder=False, n_jobs=-1)
results["xgboost"] = train_and_log(xgb, {"model":"xgboost","n_estimators":200,"max_depth":5,"learning_rate":0.1,"scale_pos_weight":round(sw,3)}, "xgboost")

lgb = LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, is_unbalance=True,
                     random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)
results["lightgbm"] = train_and_log(lgb, {"model":"lightgbm","n_estimators":200,"max_depth":5,"learning_rate":0.1,"is_unbalance":True}, "lightgbm")

compare = pd.DataFrame([{"model": k, "roc_auc": v[0], "run_id": v[1]} for k, v in results.items()])
print("\n=== Leaderboard (by AUC) ===")
print(compare.sort_values("roc_auc", ascending=False).to_string(index=False))

2026/07/26 14:33:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully registered model 'churn_model'.
2026/07/26 14:33:57 WARNING mlflow.tracking._model_registry.fluent: Run with id 273e4c0449e34598833ab670cd492147 has no artifacts at artifact path 'model', registering model based on models:/m-1355aa2aabf1493588902190dd699226 instead
Created version '1' of model 'churn_model'.


[xgboost] AUC=0.8292  recall_churn=0.73


2026/07/26 14:33:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'churn_model' already exists. Creating a new version of this model...
2026/07/26 14:34:02 WARNING mlflow.tracking._model_registry.fluent: Run with id 72e11fffb33844e48e31541a90d7908d has no artifacts at artifact path 'model', registering model based on models:/m-6cf8a9977a9c4e79b146fe06335882b1 instead


[lightgbm] AUC=0.8312  recall_churn=0.74

=== Leaderboard (by AUC) ===
   model  roc_auc                           run_id
lightgbm 0.831153 72e11fffb33844e48e31541a90d7908d
 xgboost 0.829221 273e4c0449e34598833ab670cd492147


Created version '2' of model 'churn_model'.


In [8]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
best = compare.sort_values("roc_auc", ascending=False).iloc[0]
print("🏆 Champion by AUC:", best["model"])
try:
    versions = client.search_model_versions(filter_string=f"run_id='{best.run_id}'")
    if versions:
        client.set_registered_model_alias(name="churn_model", alias="champion", version=versions[0].version)
        print(f"   alias 'champion' -> version {versions[0].version}")
except Exception as e:
    print("   (alias step skipped:", e, ")")

🏆 Champion by AUC: lightgbm
   alias 'champion' -> version 2
